## EdTech Content Data Pipeline
### AI · Data · Cloud — API Sources Collection

### Objective
Build a repeatable Data Engineering pipeline that collects, standardizes, enriches, and curates educational content about AI, Data, and Cloud from public APIs.


### 1. Setup

Import the libraries and define the API source URLs.

In [0]:
import requests
import pandas as pd
import json
import random

# Data source URLs
coursera_url = "https://api.coursera.org/api/courses.v1"
microsoft_learn_url = "https://learn.microsoft.com/api/catalog/"
github_url = "https://api.github.com/search/repositories"
youtube_url = "https://www.googleapis.com/youtube/v3"

# Define data sources
sources = {
    "Coursera": coursera_url,
    "Microsoft Learn": microsoft_learn_url,
    "GitHub": github_url,
    "YouTube": youtube_url
}

print("Data sources:")

for source, url in sources.items():
    print(f"{source}: {url}")

Data sources:
Coursera: https://api.coursera.org/api/courses.v1
Microsoft Learn: https://learn.microsoft.com/api/catalog/
GitHub: https://api.github.com/search/repositories
YouTube: https://www.googleapis.com/youtube/v3


### 2. Coursera Collection

In [0]:
# Collect 300 Coursera courses

topics = {
    "AI": ["artificial intelligence", "machine learning", "generative ai", "deep learning", "computer vision", "NLP"],
    "Data": ["data", "data analysis", "data science", "big data", "SQL", "Python"],
    "Cloud": ["cloud", "cloud computing", "AWS", "Azure", "Google Cloud"]
}

all_courses = []
counts = {"AI": 0, "Data": 0, "Cloud": 0}

for start in range(0, 24000, 100):
    response = requests.get(
        coursera_url,
        params={"start": start, "limit": 100, "fields": "id,name,slug,startDate"}
    )

    for course in response.json().get("elements", []):
        date = pd.to_datetime(course.get("startDate"), unit="ms", errors="coerce")

        if pd.isna(date) or not (2024 <= date.year <= 2026):
            continue

        for topic, keywords in topics.items():
            if counts[topic] < 100 and any(k.lower() in course["name"].lower() for k in keywords):
                course["topic"] = topic
                course["published_date"] = date

                all_courses.append(course)
                counts[topic] += 1
                break

    if all(v == 100 for v in counts.values()):
        break

print("Coursera:", len(all_courses))
print("AI:", counts["AI"], "| Data:", counts["Data"], "| Cloud:", counts["Cloud"])

Coursera: 300
AI: 100 | Data: 100 | Cloud: 100


### 3. Microsoft Learn Collection


In [0]:
# Load 300 Microsoft Learn modules

response = requests.get(
    microsoft_learn_url,
    params={"locale": "en-us"}
)

microsoft_data = response.json()
microsoft_modules = microsoft_data.get("modules", [])[:300]

counts = {"AI": 0, "Data": 0, "Cloud": 0}

for item in microsoft_modules:
    text = f"{item.get('title', '')} {item.get('summary', '')}".lower()

    if any(x in text for x in ["artificial intelligence", "machine learning", "generative ai", "copilot"]):
        counts["AI"] += 1
    elif any(x in text for x in ["data", "sql", "database", "analytics"]):
        counts["Data"] += 1
    else:
        counts["Cloud"] += 1

print("Microsoft Learn:", len(microsoft_modules))
print("AI:", counts["AI"], "| Data:", counts["Data"], "| Cloud:", counts["Cloud"])

Microsoft Learn: 300
AI: 37 | Data: 80 | Cloud: 183


### 4. GitHub Collection


In [0]:
# Collect 300 GitHub repositories

github_repositories = []
counts = {"AI": 0, "Data": 0, "Cloud": 0}

topics = {
    "AI": "artificial intelligence",
    "Data": "data engineering",
    "Cloud": "cloud computing"
}

for topic, query in topics.items():
    response = requests.get(
        github_url,
        params={
            "q": query,
            "sort": "stars",
            "order": "desc",
            "per_page": 100
        }
    )

    if response.status_code == 200:
        repositories = response.json().get("items", [])[:100]
        github_repositories.extend(repositories)
        counts[topic] = len(repositories)

print("GitHub:", len(github_repositories))
print("AI:", counts["AI"], "| Data:", counts["Data"], "| Cloud:", counts["Cloud"])

GitHub: 300
AI: 100 | Data: 100 | Cloud: 100


### 5. YouTube Collection

In [0]:
# Collect 300 YouTube educational videos

youtube_api_url = f"{youtube_url}/search"

API_KEY = "Your key"

youtube_topics = {
    "AI": ["artificial intelligence tutorial", "machine learning tutorial", "generative AI tutorial"],
    "Data": ["data science tutorial", "data analytics tutorial", "SQL data analysis"],
    "Cloud": ["cloud computing tutorial", "AWS tutorial", "Azure tutorial"]
}

youtube_videos = []
counts = {"AI": 0, "Data": 0, "Cloud": 0}
seen_ids = set()

for topic, queries in youtube_topics.items():
    for query in queries:

        if counts[topic] >= 100:
            break

        response = requests.get(
            youtube_api_url,
            params={
                "part": "snippet",
                "q": query,
                "type": "video",
                "maxResults": 50,
                "key": API_KEY
            }
        )

        data = response.json()

        for item in data.get("items", []):

            if "videoId" not in item.get("id", {}):
                continue

            video_id = item["id"]["videoId"]

            if video_id not in seen_ids:
                seen_ids.add(video_id)

                item["topic"] = topic
                youtube_videos.append(item)
                counts[topic] += 1

            if counts[topic] >= 100:
                break

print("YouTube:", len(youtube_videos))
print("AI:", counts["AI"], "| Data:", counts["Data"], "| Cloud:", counts["Cloud"])

YouTube: 300
AI: 100 | Data: 100 | Cloud: 100


### 6. Combine and Standardize

Merge all sources into one dataset and extract standardized fields: content ID, title, description, topic, category, keywords, language, source, URL, content type, and dates.

In [0]:
# Prepare and combine all resources

combined_df = all_courses + microsoft_modules + github_repositories + youtube_videos


# Extract multiple keywords from title, description, URL, and topic

import re

STOPWORDS = {
    "https", "http", "www", "com", "org", "net",
    "the", "and", "for", "with", "from", "this",
    "that", "are", "you", "your", "into", "using",
    "use", "how", "get", "getting", "learn",
    "course", "courses", "module", "video",
    "videos", "watch", "youtube"
}


def extract_keywords(title="", description="", url="", topic=""):

    text = f"{title} {description} {url} {topic}".lower()

    text = re.sub(r"[-_/?.=&%:#]+", " ", text)

    words = re.findall(r"[a-zA-Z][a-zA-Z0-9+#.]*", text)

    keywords = []

    for word in words:

        word = word.strip(".")

        if (
            len(word) >= 3
            and word not in STOPWORDS
            and word not in keywords
        ):
            keywords.append(word)

    return ", ".join(keywords[:10])


for item in combined_df:

    if "slug" in item:  # Coursera

        item["content_id"] = str(item.get("id", ""))
        item["title"] = item.get("name", "")
        item["description"] = item.get("name", "")
        item["topic"] = item.get("topic", "")

        # Category
        text = item["title"].lower()

        if any(x in text for x in ["machine learning", "deep learning"]):
            item["category"] = "Machine Learning"
        elif any(x in text for x in ["data engineering", "data science", "data analysis"]):
            item["category"] = "Data Engineering"
        elif any(x in text for x in ["cloud", "aws", "azure", "google cloud"]):
            item["category"] = "Cloud Computing"
        else:
            item["category"] = item["topic"]

        item["difficulty_level"] = random.choice(
            ["Beginner", "Intermediate", "Advanced"]
        )

        item["list of keywords"] = extract_keywords(
            title=item.get("name", ""),
            description=item.get("name", ""),
            url=f"https://www.coursera.org/learn/{item['slug']}",
            topic=item.get("topic", "")
        )

        item["language"] = "English"
        item["source"] = "Coursera"
        item["url"] = f"https://www.coursera.org/learn/{item['slug']}"

        # Content type
        item["content_type"] = "Course"

        item["published_date"] = str(item.get("published_date", ""))
        item["last_updated"] = str(item.get("published_date", ""))


    elif "uid" in item:  # Microsoft Learn

        title = item.get("title", "")
        text = f"{title} {item.get('summary', '')}".lower()

        if any(x in text for x in [
            "artificial intelligence",
            "machine learning",
            "generative ai",
            "copilot"
        ]):
            topic = "AI"

        elif any(x in text for x in [
            "data",
            "sql",
            "database",
            "analytics"
        ]):
            topic = "Data"

        else:
            topic = "Cloud"

        item["content_id"] = str(item.get("uid", ""))
        item["title"] = title
        item["description"] = item.get("summary") or title
        item["topic"] = topic

        # Category
        if any(x in text for x in [
            "machine learning",
            "artificial intelligence",
            "generative ai",
            "copilot"
        ]):
            item["category"] = "Machine Learning"
        elif any(x in text for x in [
            "data engineering",
            "data analysis",
            "data science",
            "sql",
            "database"
        ]):
            item["category"] = "Data Engineering"
        else:
            item["category"] = "Cloud Computing"

        item["difficulty_level"] = random.choice(
            ["Beginner", "Intermediate", "Advanced"]
        )

        item["keywords"] = extract_keywords(
            title=item.get("title", ""),
            description=item.get("summary", ""),
            url=item.get("url", ""),
            topic=topic
        )

        item["language"] = "English"
        item["source"] = "Microsoft Learn"
        item["url"] = item.get("url", "")

        # Content type
        item["content_type"] = "Module"

        item["published_date"] = str(item.get("last_modified", ""))
        item["last_updated"] = str(item.get("last_modified", ""))


    elif "html_url" in item:  # GitHub

        title = item.get("name", "")
        description = item.get("description") or title

        text = f"{title} {description}".lower()

        if any(x in text for x in [
            "artificial intelligence",
            "machine learning",
            "deep learning",
            "generative ai"
        ]):
            topic = "AI"

        elif any(x in text for x in [
            "data engineering",
            "data science",
            "database",
            "sql",
            "analytics"
        ]):
            topic = "Data"

        else:
            topic = "Cloud"

        item["content_id"] = str(item.get("id", ""))
        item["title"] = title
        item["description"] = description
        item["topic"] = topic

        # Category
        if any(x in text for x in [
            "machine learning",
            "artificial intelligence",
            "deep learning",
            "generative ai"
        ]):
            item["category"] = "Machine Learning"
        elif any(x in text for x in [
            "data engineering",
            "data science",
            "database",
            "sql",
            "analytics"
        ]):
            item["category"] = "Data Engineering"
        else:
            item["category"] = "Cloud Computing"

        item["difficulty_level"] = random.choice(
            ["Beginner", "Intermediate", "Advanced"]
        )

        item["keywords"] = extract_keywords(
            title=item.get("name", ""),
            description=item.get("description", ""),
            url=item.get("html_url", ""),
            topic=topic
        )

        item["language"] = item.get("language") or "English"
        item["source"] = "GitHub"
        item["url"] = item.get("html_url", "")

        # Content type
        item["content_type"] = "Repository"

        item["published_date"] = str(item.get("created_at", ""))
        item["last_updated"] = str(item.get("updated_at", ""))


    else:  # YouTube

        snippet = item.get("snippet", {})
        video_id = item.get("id", {}).get("videoId")

        title = snippet.get("title", "")
        description = snippet.get("description") or title
        text = f"{title} {description}".lower()

        item["content_id"] = str(video_id or "")
        item["title"] = title
        item["description"] = description
        item["topic"] = item.get("topic", "")

        # Category
        if any(x in text for x in [
            "machine learning",
            "deep learning",
            "artificial intelligence",
            "generative ai"
        ]):
            item["category"] = "Machine Learning"
        elif any(x in text for x in [
            "data engineering",
            "data science",
            "data analysis",
            "sql",
            "database"
        ]):
            item["category"] = "Data Engineering"
        else:
            item["category"] = "Cloud Computing"

        item["difficulty_level"] = random.choice(
            ["Beginner", "Intermediate", "Advanced"]
        )

        item["keywords"] = extract_keywords(
            title=title,
            description=description,
            url=f"https://www.youtube.com/watch?v={video_id}",
            topic=item.get("topic", "")
        )

        item["language"] = "English"
        item["source"] = "YouTube"
        item["url"] = f"https://www.youtube.com/watch?v={video_id}"

        # Content type
        item["content_type"] = "Video"

        item["published_date"] = str(snippet.get("publishedAt", ""))
        item["last_updated"] = str(snippet.get("publishedAt", ""))


print("Total records:", len(combined_df))

Total records: 1200


### 7. Final Dataset Preview

Select the required 13 columns and display the combined dataset.

In [0]:
%sql

MERGE INTO edtech.silver.youtube_videos AS t

USING (

  WITH c AS (

    SELECT

      NULLIF(TRIM(content_id), '') AS content_id,
      NULLIF(TRIM(title), '') AS title,
      TRIM(description) AS description,

      CASE
        WHEN LOWER(TRIM(topic)) = 'ai' THEN 'AI'
        WHEN LOWER(TRIM(topic)) = 'data' THEN 'Data'
        WHEN LOWER(TRIM(topic)) = 'cloud' THEN 'Cloud'
        ELSE INITCAP(TRIM(topic))
      END AS topic,

      TRIM(category) AS category,

      -- 3-tier keyword extraction: dictionary → title & description → category
      COALESCE(

        -- Tier 1: Dictionary matching
        NULLIF(CONCAT_WS(', ',

          CASE WHEN LOWER(title || ' ' || description) LIKE '%artificial intelligence%' THEN 'artificial intelligence' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%machine learning%' THEN 'machine learning' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%deep learning%' THEN 'deep learning' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%generative ai%' THEN 'generative ai' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%large language model%' THEN 'large language model' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%llm%' THEN 'llm' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%computer vision%' THEN 'computer vision' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%nlp%' THEN 'nlp' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%reinforcement learning%' THEN 'reinforcement learning' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%transformer%' THEN 'transformer' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%neural network%' THEN 'neural network' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%gpt%' THEN 'gpt' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%pytorch%' THEN 'pytorch' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%tensorflow%' THEN 'tensorflow' END,

          CASE WHEN LOWER(title || ' ' || description) LIKE '%data engineering%' THEN 'data engineering' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%data pipeline%' THEN 'data pipeline' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%data warehouse%' THEN 'data warehouse' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%data lake%' THEN 'data lake' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%etl%' THEN 'etl' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%spark%' THEN 'apache spark' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%kafka%' THEN 'apache kafka' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%sql%' THEN 'sql' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%database%' THEN 'database' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%big data%' THEN 'big data' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%databricks%' THEN 'databricks' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%postgresql%' THEN 'postgresql' END,

          CASE WHEN LOWER(title || ' ' || description) LIKE '%cloud computing%' THEN 'cloud computing' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%cloud architecture%' THEN 'cloud architecture' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%cloud security%' THEN 'cloud security' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%kubernetes%' THEN 'kubernetes' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%serverless%' THEN 'serverless' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%terraform%' THEN 'terraform' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%devops%' THEN 'devops' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%microservices%' THEN 'microservices' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%aws%' THEN 'aws' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%azure%' THEN 'azure' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%google cloud%' THEN 'google cloud' END,
          CASE WHEN LOWER(title || ' ' || description) LIKE '%docker%' THEN 'docker' END

        ), ''),

        -- Tier 2: Extract keywords from title and description
        NULLIF(
          LOWER(
            CONCAT_WS(', ',
              SLICE(
                FILTER(
                  TRANSFORM(
                    SPLIT(
                      LOWER(CONCAT_WS(' ', title, description)),
                      ' '
                    ),
                    w -> REGEXP_REPLACE(w, '[^a-z0-9]', '')
                  ),
                  w -> LENGTH(w) >= 4
                       AND w NOT IN (
                         'with','from','this','that','your','have',
                         'what','when','where','which','will','into',
                         'over','using','introduction','course','guide',
                         'tutorial','learn','basics','advanced','complete',
                         'master','part','step','about','they','their',
                         'there','these','those','then','than','also',
                         'more','some'
                       )
                ),
                1,
                5
              )
            )
          ),
          ''
        ),

        -- Tier 3: Category fallback
        LOWER(TRIM(category))

      ) AS list_of_keywords,

      TRIM(language) AS language,
      TRIM(source) AS source,
      NULLIF(TRIM(url), '') AS url,
      TRIM(content_type) AS content_type,

      INITCAP(
        NULLIF(TRIM(difficulty_level), '')
      ) AS difficulty_level,

      TRY_CAST(published_date AS TIMESTAMP) AS published_date,
      TRY_CAST(last_updated AS TIMESTAMP) AS last_updated,

      current_timestamp() AS silver_ingested_at

    FROM edtech.bronze.api_raw

    WHERE LOWER(TRIM(source)) = 'youtube'
  )

  SELECT *
  FROM c

  WHERE content_id IS NOT NULL
    AND title IS NOT NULL
    AND url IS NOT NULL

  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY content_id
    ORDER BY last_updated DESC NULLS LAST
  ) = 1

) AS s

ON t.content_id = s.content_id

WHEN MATCHED AND (
       s.last_updated > t.last_updated
    OR (t.last_updated IS NULL AND s.last_updated IS NOT NULL)
)

THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *;

content_id,title,description,topic,category,difficulty_level,list of keywords,language,source,url,content_type,published_date,last_updated
GiH_9yIGEfGUUBJ0Ws3UKQ,PostgreSQL 17: Core Concepts and Advanced SQL Techniques,PostgreSQL 17: Core Concepts and Advanced SQL Techniques,Data,Data,Intermediate,"postgresql, core, concepts, advanced, sql, techniques, coursera, packt, ojoqu, data",English,Coursera,https://www.coursera.org/learn/packt-postgresql-17-core-concepts-and-advanced-sql-techniques-ojoqu,Course,2026-05-05 13:02:01.242000,2026-05-05 13:02:01.242000
_UffpAE7EfGXWw4BmDDNwQ,"Memory, Encryption, and Protecting Data in Android","Memory, Encryption, and Protecting Data in Android",Data,Data,Advanced,"memory, encryption, protecting, data, android, coursera",English,Coursera,https://www.coursera.org/learn/memory-encryption-and-protecting-data-android,Course,2026-02-11 23:24:42.323000,2026-02-11 23:24:42.323000
CzShPlkgEfGiXg5eMLjBhw,Modern Spring Security & Cloud Integration,Modern Spring Security & Cloud Integration,Cloud,Cloud Computing,Advanced,"modern, spring, security, cloud, integration, coursera, packt",English,Coursera,https://www.coursera.org/learn/packt-modern-spring-security-and-cloud-integration,Course,2026-08-18 13:02:24.032000,2026-08-18 13:02:24.032000
yBVYC42DEfGYQQr__EfTsQ,Apply C# Fundamentals and Data Structures,Apply C# Fundamentals and Data Structures,Data,Data,Beginner,"apply, fundamentals, data, structures, coursera, csharp",English,Coursera,https://www.coursera.org/learn/apply-csharp-fundamentals-data-structures,Course,2026-09-17 02:42:35.693000,2026-09-17 02:42:35.693000
egap7M0cEe-xcgr_7YhUxQ,Modelos predictivos con Machine Learning,Modelos predictivos con Machine Learning,AI,Machine Learning,Beginner,"modelos, predictivos, con, machine, learning, coursera",English,Coursera,https://www.coursera.org/learn/modelos-predictivos-con-machine-learning,Course,2025-02-21 22:49:36.646000,2025-02-21 22:49:36.646000
z2hE2VfwEfCekhJDthqyWw,Introduction to SQL and relational databases,Introduction to SQL and relational databases,Data,Data,Intermediate,"introduction, sql, relational, databases, coursera, valencia, sandbox, data",English,Coursera,https://www.coursera.org/learn/valencia-sandbox-2-,Course,2025-09-10 16:50:54.561000,2025-09-10 16:50:54.561000
cdLlrD7FEe6U4g7a-RyA1w,Data Management and Sharing for NIH Proposals,Data Management and Sharing for NIH Proposals,Data,Data,Advanced,"data, management, sharing, nih, proposals, coursera",English,Coursera,https://www.coursera.org/learn/nih-data-sharing,Course,2024-03-15 14:03:00.463000,2024-03-15 14:03:00.463000
wBYtMroaEfCh_g7IcfE8GQ,Data Storage and Management for Big Data,Data Storage and Management for Big Data,Data,Data,Advanced,"data, storage, management, big, coursera, microsoft",English,Coursera,https://www.coursera.org/learn/microsoft-big-data-storage-management,Course,2026-01-05 17:33:55.549000,2026-01-05 17:33:55.549000
FTu5SITkEe-HnxIaoRsVow,Cybersecurity Identity and Access Solutions with Azure AD,Cybersecurity Identity and Access Solutions with Azure AD,Cloud,Cloud Computing,Advanced,"cybersecurity, identity, access, solutions, azure, coursera, cloud",English,Coursera,https://www.coursera.org/learn/cybersecurity-identity-access-solutions-azure-ad,Course,2024-10-07 21:24:09.394000,2024-10-07 21:24:09.394000
uu65YNEZEfCOyw4T7tU5XQ,Choosing your AWS identity service,Choosing your AWS identity service,Cloud,Cloud Computing,Advanced,"choosing, aws, identity, service, coursera, cloud",English,Coursera,https://www.coursera.org/learn/choosing-your-aws-identity-service,Course,2025-12-04 14:09:07.231000,2025-12-04 14:09:07.231000


### 8. Export Dataset

Download the final dataset as JSON.

In [0]:
import json, base64, os
from IPython.display import display, HTML

json_filename = 'apis_sources_1200.json'

if 'combined_df' not in globals():
    print(" combined_df not found. Run all cells first.")
else:
    records = combined_df.to_dict(orient='records')
    json_str = json.dumps(records, ensure_ascii=False, indent=2, default=str)
    
    encoded = base64.b64encode(json_str.encode('utf-8')).decode('utf-8')
    size_kb = round(len(json_str.encode('utf-8')) / 1024, 2)
    
    print(f" Records: {len(records)}")
    print(f" Size: {size_kb} KB")
    print(" Click below:")
    
    display(HTML(f"""
    <a download="{json_filename}"
       href="data:application/json;base64,{encoded}"
       style="display:inline-block;padding:14px 32px;
              background:#1E88E5;color:white;
              text-decoration:none;border-radius:8px;
              font-weight:bold;font-size:16px;">
       ⬇️ Download {json_filename}
    </a>
    """))

 Records: 1200
 Size: 1146.84 KB
 Click below:
